# WiDS Wildfire Breakthrough Portfolio

In [1]:

import os, sys, json, math, warnings, subprocess, re, hashlib
warnings.filterwarnings("ignore")

RUN_MODE = "full"   # "full" or "fast"
DO_OOF = True
AUTO_INSTALL_SKSURV = True
WORK_DIR = "/kaggle/working" if os.path.isdir("/kaggle/working") else "."
OUTPUT_PATH = os.path.join(WORK_DIR, "submission.csv")

def find_data_dir():
    candidates = [
        "/kaggle/input/competitions/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/widsworldwide-globaldathon26",
        "/kaggle/input/WiDSWorldWide_GlobalDathon26",
        "/kaggle/input/wiDSworldwide_globaldathon26",
        ".",
        "/mnt/data",
    ]
    for path in candidates:
        if all(os.path.exists(os.path.join(path, f)) for f in ["train.csv", "test.csv", "sample_submission.csv"]):
            return path
    if os.path.isdir("/kaggle/input"):
        for root, dirs, files in os.walk("/kaggle/input"):
            files = set(files)
            if {"train.csv", "test.csv", "sample_submission.csv"}.issubset(files):
                return root
    raise FileNotFoundError("Could not find competition data directory.")

DATA_DIR = find_data_dir()

def ensure_pkg(pkg, import_name=None, allow_install=True):
    name = import_name or pkg
    try:
        __import__(name)
        return True
    except Exception:
        if not allow_install:
            return False
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])
            __import__(name)
            return True
        except Exception as e:
            print(f"[WARN] package unavailable: {pkg} -> {e}")
            return False

HAS_SKSURV = ensure_pkg("scikit-survival", "sksurv", AUTO_INSTALL_SKSURV)
ensure_pkg("lightgbm", "lightgbm", True)
ensure_pkg("catboost", "catboost", True)

import numpy as np
import pandas as pd
import lightgbm as lgb
from scipy.optimize import minimize
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.isotonic import IsotonicRegression

try:
    from catboost import CatBoostClassifier
    HAS_CAT = True
except Exception:
    HAS_CAT = False

if HAS_SKSURV:
    try:
        from sksurv.util import Surv
        from sksurv.ensemble import GradientBoostingSurvivalAnalysis, RandomSurvivalForest
        from sksurv.linear_model import CoxPHSurvivalAnalysis
    except Exception:
        HAS_SKSURV = False

train_df = pd.read_csv(os.path.join(DATA_DIR, "train.csv"))
test_df = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))
sample_sub = pd.read_csv(os.path.join(DATA_DIR, "sample_submission.csv"))

print("DATA_DIR =", DATA_DIR)
print("TRAIN =", train_df.shape, "TEST =", test_df.shape, "HAS_SKSURV =", HAS_SKSURV, "HAS_CAT =", HAS_CAT)

def create_features(df, fit_df=None):
    ref = fit_df if fit_df is not None else df
    r = df.copy()

    dist = r["dist_min_ci_0_5h"].clip(lower=1)
    speed = r["closing_speed_m_per_h"]
    perims = r["num_perimeters_0_5h"]
    area_first = r["area_first_ha"]
    area_growth = r["area_growth_rate_ha_per_h"]
    radial = r["radial_growth_rate_m_per_h"].clip(lower=0)
    align = r["alignment_abs"]
    fire_radius = np.sqrt(np.maximum(area_first, 0) * 10000.0 / np.pi)
    closing_pos = speed.clip(lower=0)
    eff_close = closing_pos + radial

    r["dist_km"] = dist / 1000.0
    r["log_distance"] = np.log1p(dist)
    r["inv_distance"] = 1.0 / (dist / 1000.0 + 0.1)
    r["inv_distance_sq"] = r["inv_distance"] ** 2
    r["sqrt_distance"] = np.sqrt(dist)
    r["dist_km_sq"] = r["dist_km"] ** 2
    r["dist_km_cb"] = r["dist_km"] ** 3

    r["fire_radius_km"] = fire_radius / 1000.0
    r["radius_to_dist"] = fire_radius / dist
    r["area_to_dist_ratio"] = area_first / (dist / 1000.0 + 0.1)
    r["log_area_dist_ratio"] = np.log1p(area_first) - np.log1p(dist)

    r["has_movement"] = (perims > 1).astype(float)
    r["effective_closing_speed"] = eff_close
    r["eta_hours"] = np.where(closing_pos > 0.01, dist / closing_pos, 9999.0).clip(max=9999.0)
    r["eta_effective"] = np.where(eff_close > 0.01, dist / eff_close, 9999.0).clip(max=9999.0)
    r["log_eta"] = np.log1p(r["eta_hours"])
    r["log_eta_effective"] = np.log1p(r["eta_effective"])

    r["threat_score"] = align * speed / np.log1p(dist)
    r["threat_score_eff"] = align * eff_close / np.log1p(dist)
    r["threat_score_sq"] = r["threat_score_eff"] ** 2
    r["fire_urgency"] = perims * speed
    r["growth_intensity"] = area_growth * perims
    r["speed_alignment"] = speed * align
    r["effective_alignment"] = eff_close * align

    r["projected_advance_pos"] = np.maximum(r["projected_advance_m"], 0)
    r["closing_distance_pos"] = np.maximum(-r["dist_change_ci_0_5h"], 0)
    r["slope_toward"] = np.maximum(-r["dist_slope_ci_0_5h"], 0)
    r["accel_toward"] = np.maximum(-r["dist_accel_m_per_h2"], 0)

    r["zone_3km"] = (dist < 3000).astype(float)
    r["zone_near"] = (dist < 5000).astype(float)
    r["zone_warning"] = ((dist >= 5000) & (dist < 10000)).astype(float)
    r["zone_far"] = (dist >= 10000).astype(float)
    r["zone_20km"] = (dist < 20000).astype(float)

    r["is_summer"] = r["event_start_month"].isin([6, 7, 8]).astype(float)
    r["is_afternoon"] = ((r["event_start_hour"] >= 12) & (r["event_start_hour"] < 20)).astype(float)
    r["is_night"] = ((r["event_start_hour"] <= 6) | (r["event_start_hour"] >= 21)).astype(float)

    r["hour_sin"] = np.sin(2 * np.pi * r["event_start_hour"] / 24.0)
    r["hour_cos"] = np.cos(2 * np.pi * r["event_start_hour"] / 24.0)
    r["month_sin"] = np.sin(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["month_cos"] = np.cos(2 * np.pi * (r["event_start_month"] - 1) / 12.0)
    r["dow_sin"] = np.sin(2 * np.pi * r["event_start_dayofweek"] / 7.0)
    r["dow_cos"] = np.cos(2 * np.pi * r["event_start_dayofweek"] / 7.0)

    def pct_rank(values, ref_values):
        ref_values = np.asarray(ref_values, dtype=float)
        if ref_values.size == 0:
            return np.zeros(len(values), dtype=float)
        ref_sorted = np.sort(ref_values)
        return np.searchsorted(ref_sorted, np.asarray(values, dtype=float), side="right") / ref_sorted.size

    ref_dist = ref["dist_min_ci_0_5h"].clip(lower=1).values
    ref_eff = (ref["closing_speed_m_per_h"].clip(lower=0).values +
               ref["radial_growth_rate_m_per_h"].clip(lower=0).values)
    ref_threat = (
        ref["alignment_abs"].values * ref_eff /
        np.log1p(ref["dist_min_ci_0_5h"].clip(lower=1).values)
    )

    r["dist_rank_global"] = pct_rank(dist.values, ref_dist)
    r["eff_rank_global"] = pct_rank(eff_close.values, ref_eff)
    r["threat_rank_global"] = pct_rank(r["threat_score_eff"].values, ref_threat)

    ref_near = ref["dist_min_ci_0_5h"].clip(lower=1).values < 5000
    cur_near = dist.values < 5000
    near_speed_rank = np.zeros(len(r), dtype=float)
    far_threat_rank = np.zeros(len(r), dtype=float)
    near_ref_eff = ref_eff[ref_near]
    far_ref_threat = ref_threat[~ref_near]
    if cur_near.any():
        near_speed_rank[cur_near] = pct_rank(eff_close.values[cur_near], near_ref_eff)
    if (~cur_near).any():
        far_threat_rank[~cur_near] = pct_rank(r["threat_score_eff"].values[~cur_near], far_ref_threat)
    r["near_speed_rank"] = near_speed_rank
    r["far_threat_rank"] = far_threat_rank

    drop_cols = [
        "relative_growth_0_5h",
        "projected_advance_m",
        "centroid_displacement_m",
        "centroid_speed_m_per_h",
        "closing_speed_abs_m_per_h",
        "area_growth_abs_0_5h",
    ]
    r = r.drop(columns=[c for c in drop_cols if c in r.columns])
    r = r.replace([np.inf, -np.inf], np.nan).fillna(0)
    return r

train_proc = create_features(train_df, fit_df=train_df)
test_proc = create_features(test_df, fit_df=train_df)

time_values = train_df["time_to_hit_hours"].values
event_values = train_df["event"].astype(int).values
dist_train = train_df["dist_min_ci_0_5h"].values
dist_test = test_df["dist_min_ci_0_5h"].values

def compute_c_index(time, event, risk):
    n = len(time)
    conc = 0.0
    comp = 0.0
    for i in range(n):
        if event[i] != 1:
            continue
        for j in range(n):
            if i == j or time[i] >= time[j]:
                continue
            comp += 1.0
            if risk[i] > risk[j]:
                conc += 1.0
            elif risk[i] == risk[j]:
                conc += 0.5
    return conc / comp if comp else 0.5

def compute_brier(time, event, prob, horizon):
    valid = ~((event == 0) & (time < horizon))
    if valid.sum() == 0:
        return 0.25
    y_true = ((event == 1) & (time <= horizon)).astype(float)[valid]
    prob = np.clip(np.asarray(prob, dtype=float)[valid], 0.0, 1.0)
    return float(np.mean((prob - y_true) ** 2))

def compute_hybrid_score(time, event, p24, p48, p72):
    risk = 0.3 * p24 + 0.4 * p48 + 0.3 * p72
    c_idx = compute_c_index(time, event, risk)
    b24 = compute_brier(time, event, p24, 24)
    b48 = compute_brier(time, event, p48, 48)
    b72 = compute_brier(time, event, p72, 72)
    wb = 0.3 * b24 + 0.4 * b48 + 0.3 * b72
    return 0.3 * c_idx + 0.7 * (1.0 - wb), c_idx, wb, b24, b48, b72

def enforce_monotonicity(preds):
    out = np.clip(np.asarray(preds, dtype=float), 0.0, 1.0)
    for i in range(1, out.shape[1]):
        out[:, i] = np.maximum(out[:, i], out[:, i - 1])
    return out

def make_binary_target(time_vals, event_vals, horizon):
    unknown = (event_vals == 0) & (time_vals < horizon)
    y = ((event_vals == 1) & (time_vals <= horizon)).astype(int)
    return y, ~unknown

def compute_ipcw_weights(times, events, horizon):
    times = np.asarray(times, dtype=float)
    events = np.asarray(events, dtype=int)
    unique_t = np.sort(np.unique(times))
    surv = np.ones(len(unique_t), dtype=float)
    for i, t in enumerate(unique_t):
        at_risk = np.sum(times >= t)
        cens = np.sum((times == t) & (events == 0))
        if at_risk > 0:
            surv[i] = 1.0 - cens / at_risk
        if i > 0:
            surv[i] *= surv[i - 1]
    def G(t):
        idx = np.searchsorted(unique_t, t, side="right") - 1
        if idx >= 0:
            return max(float(surv[idx]), 0.01)
        return 1.0
    weights = np.ones(len(times), dtype=float)
    for i in range(len(times)):
        if events[i] == 1 and times[i] <= horizon:
            weights[i] = 1.0 / G(times[i])
        elif times[i] >= horizon:
            weights[i] = 1.0 / G(horizon)
        else:
            weights[i] = 0.0
    return weights

def fit_with_optional_weight(model, X, y, sample_weight=None):
    if sample_weight is None:
        model.fit(X, y)
        return model
    if isinstance(model, Pipeline):
        last_name = model.steps[-1][0]
        try:
            model.fit(X, y, **{f"{last_name}__sample_weight": sample_weight})
            return model
        except Exception:
            model.fit(X, y)
            return model
    try:
        model.fit(X, y, sample_weight=sample_weight)
        return model
    except Exception:
        model.fit(X, y)
        return model

def safe_predict_proba(model, X):
    p = model.predict_proba(X)
    if isinstance(p, list):
        p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else p

def soft_gate(dist_m, center, width):
    z = np.clip((dist_m - center) / max(width, 1.0), -60, 60)
    return 1.0 / (1.0 + np.exp(z))

def repeated_isotonic(train_signal, test_signal, y, mask, seeds, n_splits=5):
    train_signal = np.asarray(train_signal, dtype=float)
    test_signal = np.asarray(test_signal, dtype=float)
    valid_idx = np.where(mask)[0]
    yv = y[valid_idx]
    oof = np.zeros(len(train_signal), dtype=float)
    cnt = np.zeros(len(train_signal), dtype=float)
    full_fill = np.zeros(len(train_signal), dtype=float)
    test_pred = np.zeros(len(test_signal), dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(valid_idx, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            ir = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
            ir.fit(train_signal[tr_idx], y[tr_idx])
            oof[va_idx] += ir.predict(train_signal[va_idx])
            cnt[va_idx] += 1.0

        ir_full = IsotonicRegression(y_min=0.0, y_max=1.0, out_of_bounds="clip")
        ir_full.fit(train_signal[valid_idx], yv)
        full_fill += ir_full.predict(train_signal)
        test_pred += ir_full.predict(test_signal)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

def repeated_binary_model(X_train, X_test, y, mask, build_model, seeds, n_splits=5, horizon=None, use_ipcw=False):
    n_train = len(X_train)
    n_test = len(X_test)
    valid_idx = np.where(mask)[0]
    Xv = X_train.iloc[valid_idx]
    yv = y[valid_idx]
    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            model = build_model(seed)
            sw = None
            if use_ipcw and horizon is not None:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        sw_full = None
        if use_ipcw and horizon is not None:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        full_fill += safe_predict_proba(model_full, X_train)
        test_pred += safe_predict_proba(model_full, X_test)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

def repeated_binary_model_subset(X_train, X_test, y, subset_mask, build_model, seeds, n_splits=5, horizon=None, use_ipcw=False):
    n_train = len(X_train)
    n_test = len(X_test)
    valid_idx = np.where(subset_mask)[0]
    if len(valid_idx) < max(12, n_splits + 2):
        return np.full(n_train, 0.5, dtype=float), np.full(n_test, 0.5, dtype=float)
    Xv = X_train.iloc[valid_idx]
    yv = y[valid_idx]
    if len(np.unique(yv)) < 2:
        p = float(np.mean(yv))
        return np.full(n_train, p, dtype=float), np.full(n_test, p, dtype=float)

    oof = np.zeros(n_train, dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    full_fill = np.zeros(n_train, dtype=float)
    test_pred = np.zeros(n_test, dtype=float)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        for tr_sub, va_sub in cv.split(Xv, yv):
            tr_idx = valid_idx[tr_sub]
            va_idx = valid_idx[va_sub]
            model = build_model(seed)
            sw = None
            if use_ipcw and horizon is not None:
                sw = compute_ipcw_weights(time_values[tr_idx], event_values[tr_idx], horizon)
            fit_with_optional_weight(model, X_train.iloc[tr_idx], y[tr_idx], sw)
            oof[va_idx] += safe_predict_proba(model, X_train.iloc[va_idx])
            cnt[va_idx] += 1.0

        model_full = build_model(seed)
        sw_full = None
        if use_ipcw and horizon is not None:
            sw_full = compute_ipcw_weights(time_values[valid_idx], event_values[valid_idx], horizon)
        fit_with_optional_weight(model_full, Xv, yv, sw_full)
        full_fill += safe_predict_proba(model_full, X_train)
        test_pred += safe_predict_proba(model_full, X_test)

    full_fill /= len(seeds)
    test_pred /= len(seeds)
    nz = cnt > 0
    oof[nz] /= cnt[nz]
    oof[~nz] = full_fill[~nz]
    return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

HORIZONS = [12, 24, 48, 72]

if HAS_SKSURV:
    def get_surv_predictions(model, X):
        surv_fns = model.predict_survival_function(X)
        preds = np.empty((len(surv_fns), len(HORIZONS)), dtype=float)
        for i, fn in enumerate(surv_fns):
            try:
                t_min, t_max = fn.domain
            except Exception:
                t_min, t_max = fn.x[0], fn.x[-1]
            grid = np.clip(HORIZONS, t_min, t_max)
            preds[i, :] = 1.0 - fn(grid)
        return np.clip(preds, 0.0, 1.0)

    y_surv = Surv.from_arrays(
        event=train_df["event"].astype(bool).values,
        time=train_df["time_to_hit_hours"].values,
    )

    def repeated_survival_model(X_train, X_test, build_model, seeds, n_splits=5):
        n_train = len(X_train)
        n_test = len(X_test)
        oof = np.zeros((n_train, 4), dtype=float)
        cnt = np.zeros(n_train, dtype=float)
        full_fill = np.zeros((n_train, 4), dtype=float)
        test_pred = np.zeros((n_test, 4), dtype=float)

        for seed in seeds:
            cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
            seed_test = np.zeros((n_test, 4), dtype=float)

            for tr_idx, va_idx in cv.split(X_train, event_values):
                model = build_model(seed)
                try:
                    model.fit(X_train.iloc[tr_idx], y_surv[tr_idx])
                    oof[va_idx] += get_surv_predictions(model, X_train.iloc[va_idx])
                    cnt[va_idx] += 1.0
                    seed_test += get_surv_predictions(model, X_test) / n_splits
                except Exception:
                    oof[va_idx] += 0.5
                    cnt[va_idx] += 1.0
                    seed_test += 0.5 / n_splits

            test_pred += seed_test / len(seeds)

            model_full = build_model(seed)
            try:
                model_full.fit(X_train, y_surv)
                full_fill += get_surv_predictions(model_full, X_train)
            except Exception:
                full_fill += 0.5

        full_fill /= len(seeds)
        nz = cnt > 0
        oof[nz] /= cnt[nz, None]
        oof[~nz] = full_fill[~nz]
        return np.clip(oof, 0.0, 1.0), np.clip(test_pred, 0.0, 1.0)

FAST_SEEDS = (123, 456, 789)
BASE_SEEDS = (123, 456, 789, 777, 666, 1511, 1523, 2025, 2026, 2033, 279, 239, 70, 77, 31, 2024)
GBSA_SEEDS_FULL = BASE_SEEDS + (2077, 3077, 4640, 841, 7755, 8525, 2701, 8817)
COX_SEEDS_FULL = BASE_SEEDS[:12]
RSF_SEEDS_FULL = BASE_SEEDS[:10]
TREE_SEEDS_FULL = BASE_SEEDS + (2077, 3077, 123456, 654321)
CAT_SEEDS_FULL = BASE_SEEDS[:8]

def choose_seeds(full_seeds):
    return FAST_SEEDS if RUN_MODE == "fast" else tuple(full_seeds)

RAW_FEATURES = [c for c in train_df.columns if c not in ["event_id", "event", "time_to_hit_hours"]]

COX_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "fire_radius_km",
    "num_perimeters_0_5h", "has_movement",
    "near_speed_rank", "far_threat_rank",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "is_summer", "is_afternoon", "hour_sin", "hour_cos",
    "zone_near", "zone_far",
]
COX_FEATURES = [c for c in COX_FEATURES if c in train_proc.columns]

NEAR_LGB_FEATURES = [
    "closing_speed_m_per_h", "radial_growth_rate_m_per_h",
    "alignment_abs", "num_perimeters_0_5h", "area_growth_rate_ha_per_h",
    "eta_effective", "log_eta_effective", "dist_km", "threat_score_eff",
    "near_speed_rank", "event_start_hour", "is_afternoon", "fire_urgency",
    "area_first_ha", "fire_radius_km", "low_temporal_resolution_0_5h",
    "dt_first_last_0_5h",
]
NEAR_LGB_FEATURES = [c for c in NEAR_LGB_FEATURES if c in train_proc.columns]

FAR_LGB_FEATURES = [
    "dist_km", "log_distance", "inv_distance",
    "closing_speed_m_per_h", "alignment_abs",
    "threat_score_eff", "log_eta_effective", "eta_effective",
    "area_to_dist_ratio", "num_perimeters_0_5h",
    "far_threat_rank", "is_summer", "zone_far",
    "low_temporal_resolution_0_5h", "dt_first_last_0_5h",
    "log1p_growth", "dist_fit_r2_0_5h",
]
FAR_LGB_FEATURES = [c for c in FAR_LGB_FEATURES if c in train_proc.columns]

GLOBAL_TREE_FEATURES = sorted(set(
    NEAR_LGB_FEATURES + FAR_LGB_FEATURES + [
        "inv_distance_sq", "sqrt_distance", "threat_score", "growth_intensity",
        "speed_alignment", "effective_alignment", "zone_warning",
        "hour_sin", "hour_cos", "month_sin", "month_cos", "dow_sin", "dow_cos",
        "projected_advance_pos", "closing_distance_pos", "slope_toward",
        "log_area_dist_ratio", "fire_radius_km", "log1p_area_first",
        "dist_rank_global", "eff_rank_global", "threat_rank_global",
    ]
))
GLOBAL_TREE_FEATURES = [c for c in GLOBAL_TREE_FEATURES if c in train_proc.columns]

def build_lgb_global(seed):
    return lgb.LGBMClassifier(
        n_estimators=320,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=12,
        subsample=0.82,
        colsample_bytree=0.78,
        reg_alpha=0.35,
        reg_lambda=2.0,
        random_state=seed,
        objective="binary",
        verbosity=-1,
    )

def build_lgb_near(seed):
    return lgb.LGBMClassifier(
        n_estimators=260,
        learning_rate=0.035,
        num_leaves=15,
        min_child_samples=10,
        subsample=0.82,
        colsample_bytree=0.82,
        reg_alpha=0.20,
        reg_lambda=1.8,
        random_state=seed,
        objective="binary",
        verbosity=-1,
    )

def build_lgb_far(seed):
    return lgb.LGBMClassifier(
        n_estimators=280,
        learning_rate=0.03,
        num_leaves=15,
        min_child_samples=14,
        subsample=0.85,
        colsample_bytree=0.78,
        reg_alpha=0.45,
        reg_lambda=2.2,
        random_state=seed,
        objective="binary",
        verbosity=-1,
    )

def build_cat_global(seed):
    if not HAS_CAT:
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=0.75, max_iter=800))
        ])
    return CatBoostClassifier(
        iterations=420,
        depth=5,
        learning_rate=0.03,
        loss_function="Logloss",
        eval_metric="Logloss",
        random_seed=seed,
        verbose=False,
        l2_leaf_reg=5.0,
        subsample=0.85,
        bootstrap_type="Bernoulli",
        allow_writing_files=False,
    )

def build_lr(seed):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("lr", LogisticRegression(C=0.75, max_iter=1000))
    ])

def build_gbsa(seed):
    return GradientBoostingSurvivalAnalysis(
        n_estimators=300,
        learning_rate=0.03,
        max_depth=3,
        min_samples_split=10,
        min_samples_leaf=5,
        subsample=0.85,
        random_state=seed,
    )

def build_rsf(seed):
    return RandomSurvivalForest(
        n_estimators=300,
        min_samples_split=10,
        min_samples_leaf=4,
        max_features="sqrt",
        n_jobs=-1,
        random_state=seed,
    )

def build_cox(seed):
    return Pipeline([
        ("scaler", StandardScaler()),
        ("cox", CoxPHSurvivalAnalysis(alpha=0.10))
    ])

def optimize_blend(P, y, prior=None, reg=0.03, shrink=0.35):
    P = np.asarray(P, dtype=float)
    y = np.asarray(y, dtype=float)
    m = P.shape[1]
    if prior is None:
        prior = np.full(m, 1.0 / m, dtype=float)
    prior = np.asarray(prior, dtype=float)
    prior = np.clip(prior, 1e-9, None)
    prior = prior / prior.sum()

    def unpack(z):
        z = np.asarray(z, dtype=float)
        z = z - np.max(z)
        w = np.exp(z)
        w = w / np.sum(w)
        return w

    def objective(z):
        w = unpack(z)
        pred = np.clip(P @ w, 0.0, 1.0)
        brier = np.mean((pred - y) ** 2)
        return float(brier + reg * np.sum((w - prior) ** 2))

    x0 = np.log(prior)
    try:
        res = minimize(objective, x0=x0, method="L-BFGS-B")
        w_opt = unpack(res.x)
    except Exception:
        w_opt = prior.copy()
    w_final = shrink * prior + (1.0 - shrink) * w_opt
    w_final = np.clip(w_final, 0.0, None)
    if w_final.sum() <= 0:
        w_final = prior.copy()
    else:
        w_final = w_final / w_final.sum()
    return w_final, w_opt

weights_log = {}

anchor_train_signal = (
    0.55 * train_proc["threat_rank_global"].values
    + 0.25 * train_proc["eff_rank_global"].values
    + 0.20 * (1.0 - train_proc["dist_rank_global"].values)
)
anchor_test_signal = (
    0.55 * test_proc["threat_rank_global"].values
    + 0.25 * test_proc["eff_rank_global"].values
    + 0.20 * (1.0 - test_proc["dist_rank_global"].values)
)
timing_train_signal = (
    -np.log1p(train_proc["eta_effective"].values)
    + 0.18 * train_proc["alignment_abs"].values
    + 0.10 * train_proc["near_speed_rank"].values
    + 0.08 * train_proc["threat_rank_global"].values
    + 0.06 * train_proc["dist_fit_r2_0_5h"].values
)
timing_test_signal = (
    -np.log1p(test_proc["eta_effective"].values)
    + 0.18 * test_proc["alignment_abs"].values
    + 0.10 * test_proc["near_speed_rank"].values
    + 0.08 * test_proc["threat_rank_global"].values
    + 0.06 * test_proc["dist_fit_r2_0_5h"].values
)

anchor_oof, anchor_test = {}, {}
timing_oof, timing_test = {}, {}
for h in [12, 24, 48]:
    y_h, mask_h = make_binary_target(time_values, event_values, h)
    anchor_oof[h], anchor_test[h] = repeated_isotonic(anchor_train_signal, anchor_test_signal, y_h, mask_h, choose_seeds(BASE_SEEDS[:6]), n_splits=5)
    timing_oof[h], timing_test[h] = repeated_isotonic(timing_train_signal, timing_test_signal, y_h, mask_h, choose_seeds(BASE_SEEDS[:6]), n_splits=5)

base_oof = {"lgb": {}, "cat": {}, "lr": {}, "cox": {}, "gbsa": {}, "rsf": {}}
base_test = {"lgb": {}, "cat": {}, "lr": {}, "cox": {}, "gbsa": {}, "rsf": {}}
near_lgb_oof, near_lgb_test = {}, {}
far_lgb_oof, far_lgb_test = {}, {}
near_cat_oof, near_cat_test = {}, {}
far_cat_oof, far_cat_test = {}, {}

for h in [12, 24, 48]:
    y_h, mask_h = make_binary_target(time_values, event_values, h)

    base_oof["lgb"][h], base_test["lgb"][h] = repeated_binary_model(
        train_proc[GLOBAL_TREE_FEATURES], test_proc[GLOBAL_TREE_FEATURES], y_h, mask_h,
        build_lgb_global, choose_seeds(TREE_SEEDS_FULL[:8]), n_splits=5, horizon=h, use_ipcw=True
    )
    base_oof["cat"][h], base_test["cat"][h] = repeated_binary_model(
        train_proc[GLOBAL_TREE_FEATURES], test_proc[GLOBAL_TREE_FEATURES], y_h, mask_h,
        build_cat_global, choose_seeds(CAT_SEEDS_FULL), n_splits=5, horizon=h, use_ipcw=True
    )
    base_oof["lr"][h], base_test["lr"][h] = repeated_binary_model(
        train_proc[COX_FEATURES], test_proc[COX_FEATURES], y_h, mask_h,
        build_lr, choose_seeds(BASE_SEEDS[:6]), n_splits=5, horizon=h, use_ipcw=True
    )

    near_mask_h = mask_h & (dist_train < 5000)
    far_mask_h = mask_h & ~(dist_train < 5000)

    near_lgb_oof[h], near_lgb_test[h] = repeated_binary_model_subset(
        train_proc[NEAR_LGB_FEATURES], test_proc[NEAR_LGB_FEATURES], y_h, near_mask_h,
        build_lgb_near, choose_seeds(TREE_SEEDS_FULL[:6]), n_splits=5, horizon=h, use_ipcw=True
    )
    far_lgb_oof[h], far_lgb_test[h] = repeated_binary_model_subset(
        train_proc[FAR_LGB_FEATURES], test_proc[FAR_LGB_FEATURES], y_h, far_mask_h,
        build_lgb_far, choose_seeds(TREE_SEEDS_FULL[:6]), n_splits=5, horizon=h, use_ipcw=True
    )
    near_cat_oof[h], near_cat_test[h] = repeated_binary_model_subset(
        train_proc[NEAR_LGB_FEATURES], test_proc[NEAR_LGB_FEATURES], y_h, near_mask_h,
        build_cat_global, choose_seeds(CAT_SEEDS_FULL[:6]), n_splits=5, horizon=h, use_ipcw=True
    )
    far_cat_oof[h], far_cat_test[h] = repeated_binary_model_subset(
        train_proc[FAR_LGB_FEATURES], test_proc[FAR_LGB_FEATURES], y_h, far_mask_h,
        build_cat_global, choose_seeds(CAT_SEEDS_FULL[:6]), n_splits=5, horizon=h, use_ipcw=True
    )

if HAS_SKSURV:
    base_oof["cox"], base_test["cox"] = {}, {}
    base_oof["gbsa"], base_test["gbsa"] = {}, {}
    base_oof["rsf"], base_test["rsf"] = {}, {}

    cox_oof_all, cox_test_all = repeated_survival_model(
        train_proc[COX_FEATURES], test_proc[COX_FEATURES], build_cox, choose_seeds(COX_SEEDS_FULL), n_splits=5
    )
    gbsa_oof_all, gbsa_test_all = repeated_survival_model(
        train_proc[GLOBAL_TREE_FEATURES], test_proc[GLOBAL_TREE_FEATURES], build_gbsa, choose_seeds(GBSA_SEEDS_FULL), n_splits=5
    )
    rsf_oof_all, rsf_test_all = repeated_survival_model(
        train_proc[GLOBAL_TREE_FEATURES], test_proc[GLOBAL_TREE_FEATURES], build_rsf, choose_seeds(RSF_SEEDS_FULL), n_splits=5
    )

    for col_idx, h in enumerate([12, 24, 48, 72]):
        base_oof["cox"][h] = cox_oof_all[:, col_idx]
        base_test["cox"][h] = cox_test_all[:, col_idx]
        base_oof["gbsa"][h] = gbsa_oof_all[:, col_idx]
        base_test["gbsa"][h] = gbsa_test_all[:, col_idx]
        base_oof["rsf"][h] = rsf_oof_all[:, col_idx]
        base_test["rsf"][h] = rsf_test_all[:, col_idx]

def assemble_candidates(h, region, split, prior_names):
    names = []
    arrs = []

    def add(name, values):
        if values is None:
            return
        arr = np.asarray(values, dtype=float)
        if np.all(np.isfinite(arr)):
            names.append(name)
            arrs.append(np.clip(arr, 0.0, 1.0))

    if split == "oof":
        add("anchor", anchor_oof[h])
        add("timing", timing_oof[h])
        add("lgb", base_oof["lgb"][h])
        add("cat", base_oof["cat"][h])
        add("lr", base_oof["lr"][h])
        add("near_lgb", near_lgb_oof[h] if region == "near" else None)
        add("near_cat", near_cat_oof[h] if region == "near" else None)
        add("far_lgb", far_lgb_oof[h] if region == "far" else None)
        add("far_cat", far_cat_oof[h] if region == "far" else None)
        if HAS_SKSURV:
            add("cox", base_oof["cox"][h])
            add("gbsa", base_oof["gbsa"][h])
            add("rsf", base_oof["rsf"][h])
    else:
        add("anchor", anchor_test[h])
        add("timing", timing_test[h])
        add("lgb", base_test["lgb"][h])
        add("cat", base_test["cat"][h])
        add("lr", base_test["lr"][h])
        add("near_lgb", near_lgb_test[h] if region == "near" else None)
        add("near_cat", near_cat_test[h] if region == "near" else None)
        add("far_lgb", far_lgb_test[h] if region == "far" else None)
        add("far_cat", far_cat_test[h] if region == "far" else None)
        if HAS_SKSURV:
            add("cox", base_test["cox"][h])
            add("gbsa", base_test["gbsa"][h])
            add("rsf", base_test["rsf"][h])

    if not arrs:
        arrs = [np.full(len(train_df) if split == "oof" else len(test_df), 0.5, dtype=float)]
        names = ["fallback"]

    P = np.column_stack(arrs)
    prior = np.asarray([prior_names.get(name, 0.0) for name in names], dtype=float)
    if prior.sum() <= 0:
        prior = np.full(len(names), 1.0 / len(names), dtype=float)
    else:
        prior = prior / prior.sum()
    return names, P, prior

PRIORS = {
    (12, "near"): {"gbsa": 0.18, "cox": 0.10, "lgb": 0.16, "cat": 0.14, "lr": 0.08, "near_lgb": 0.14, "near_cat": 0.10, "anchor": 0.05, "timing": 0.05},
    (12, "far"): {"gbsa": 0.22, "cox": 0.12, "lgb": 0.15, "cat": 0.10, "lr": 0.07, "far_lgb": 0.14, "far_cat": 0.10, "anchor": 0.05, "timing": 0.05},
    (24, "near"): {"gbsa": 0.20, "cox": 0.12, "lgb": 0.14, "cat": 0.12, "lr": 0.06, "near_lgb": 0.16, "near_cat": 0.10, "anchor": 0.05, "timing": 0.05},
    (24, "far"): {"gbsa": 0.22, "cox": 0.14, "lgb": 0.14, "cat": 0.10, "lr": 0.06, "far_lgb": 0.16, "far_cat": 0.08, "anchor": 0.05, "timing": 0.05},
    (48, "near"): {"gbsa": 0.18, "cox": 0.12, "lgb": 0.12, "cat": 0.10, "lr": 0.05, "near_lgb": 0.18, "near_cat": 0.10, "anchor": 0.06, "timing": 0.09},
    (48, "far"): {"gbsa": 0.16, "cox": 0.14, "lgb": 0.10, "cat": 0.08, "lr": 0.05, "far_lgb": 0.20, "far_cat": 0.08, "anchor": 0.08, "timing": 0.11},
}

BLEND_CFG = {
    (12, "near"): {"reg": 0.040, "shrink": 0.35},
    (12, "far"): {"reg": 0.045, "shrink": 0.40},
    (24, "near"): {"reg": 0.030, "shrink": 0.32},
    (24, "far"): {"reg": 0.035, "shrink": 0.35},
    (48, "near"): {"reg": 0.030, "shrink": 0.30},
    (48, "far"): {"reg": 0.030, "shrink": 0.28},
}

GATE_CFG = {
    12: (4500.0, 1200.0),
    24: (5200.0, 1600.0),
    48: (6500.0, 2200.0),
}

oof_final = np.zeros((len(train_df), 4), dtype=float)
test_final = np.zeros((len(test_df), 4), dtype=float)

for col_idx, h in enumerate([12, 24, 48]):
    center, width = GATE_CFG[h]
    gate_train = soft_gate(dist_train, center, width)
    gate_test = soft_gate(dist_test, center, width)

    y_h, mask_h = make_binary_target(time_values, event_values, h)

    names_n, Pn_oof, prior_vec_n = assemble_candidates(h, "near", "oof", PRIORS[(h, "near")])
    _, Pn_test, _ = assemble_candidates(h, "near", "test", PRIORS[(h, "near")])
    near_mask = mask_h & (dist_train < 5000)
    if near_mask.sum() >= max(8, len(names_n) + 2):
        w_n, opt_n = optimize_blend(
            Pn_oof[near_mask], y_h[near_mask], prior_vec_n,
            BLEND_CFG[(h, "near")]["reg"], BLEND_CFG[(h, "near")]["shrink"]
        )
    else:
        w_n, opt_n = prior_vec_n.copy(), prior_vec_n.copy()
    pred_near_oof = np.clip(Pn_oof @ w_n, 0.0, 1.0)
    pred_near_test = np.clip(Pn_test @ w_n, 0.0, 1.0)

    names_f, Pf_oof, prior_vec_f = assemble_candidates(h, "far", "oof", PRIORS[(h, "far")])
    _, Pf_test, _ = assemble_candidates(h, "far", "test", PRIORS[(h, "far")])
    far_mask = mask_h & ~(dist_train < 5000)
    if far_mask.sum() >= max(8, len(names_f) + 2):
        w_f, opt_f = optimize_blend(
            Pf_oof[far_mask], y_h[far_mask], prior_vec_f,
            BLEND_CFG[(h, "far")]["reg"], BLEND_CFG[(h, "far")]["shrink"]
        )
    else:
        w_f, opt_f = prior_vec_f.copy(), prior_vec_f.copy()
    pred_far_oof = np.clip(Pf_oof @ w_f, 0.0, 1.0)
    pred_far_test = np.clip(Pf_test @ w_f, 0.0, 1.0)

    oof_final[:, col_idx] = gate_train * pred_near_oof + (1.0 - gate_train) * pred_far_oof
    test_final[:, col_idx] = gate_test * pred_near_test + (1.0 - gate_test) * pred_far_test

    weights_log[f"{h}_near"] = {
        "names": names_n,
        "prior": [float(x) for x in prior_vec_n],
        "opt": [float(x) for x in opt_n],
        "final": [float(x) for x in w_n],
    }
    weights_log[f"{h}_far"] = {
        "names": names_f,
        "prior": [float(x) for x in prior_vec_f],
        "opt": [float(x) for x in opt_f],
        "final": [float(x) for x in w_f],
    }

oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0
oof_final = enforce_monotonicity(oof_final)
test_final = enforce_monotonicity(test_final)

exp_oof_final = oof_final.copy()
exp_test_final = test_final.copy()

if HAS_SKSURV:
    gbsa_core_oof = np.column_stack([
        base_oof["gbsa"][12],
        np.clip(base_oof["gbsa"][24] ** 1.10, 0.0, 1.0),
        base_oof["gbsa"][48],
        np.ones(len(train_df), dtype=float),
    ])
    gbsa_core_test = np.column_stack([
        base_test["gbsa"][12],
        np.clip(base_test["gbsa"][24] ** 1.10, 0.0, 1.0),
        base_test["gbsa"][48],
        np.ones(len(test_df), dtype=float),
    ])
    hard_near_train = dist_train < 5000
    hard_near_test = dist_test < 5000

    core_oof_final = np.zeros((len(train_df), 4), dtype=float)
    core_test_final = np.zeros((len(test_df), 4), dtype=float)

    near12_oof = near_lgb_oof.get(24, anchor_oof[12])
    near12_test = near_lgb_test.get(24, anchor_test[12])

    core_oof_final[:, 0] = np.where(
        hard_near_train,
        0.76 * gbsa_core_oof[:, 0] + 0.12 * base_oof["cox"][12] + 0.02 * base_oof["rsf"][12] + 0.10 * near12_oof,
        gbsa_core_oof[:, 0]
    )
    core_test_final[:, 0] = np.where(
        hard_near_test,
        0.76 * gbsa_core_test[:, 0] + 0.12 * base_test["cox"][12] + 0.02 * base_test["rsf"][12] + 0.10 * near12_test,
        gbsa_core_test[:, 0]
    )

    core_oof_final[:, 1] = np.where(
        hard_near_train,
        0.82 * gbsa_core_oof[:, 1] + 0.14 * base_oof["cox"][24] + 0.02 * base_oof["rsf"][24] + 0.02 * near_lgb_oof[24],
        0.62 * gbsa_core_oof[:, 1] + 0.25 * base_oof["cox"][24] + 0.06 * base_oof["rsf"][24] + 0.07 * far_lgb_oof[24]
    )
    core_test_final[:, 1] = np.where(
        hard_near_test,
        0.82 * gbsa_core_test[:, 1] + 0.14 * base_test["cox"][24] + 0.02 * base_test["rsf"][24] + 0.02 * near_lgb_test[24],
        0.62 * gbsa_core_test[:, 1] + 0.25 * base_test["cox"][24] + 0.06 * base_test["rsf"][24] + 0.07 * far_lgb_test[24]
    )

    core_oof_final[:, 2] = np.where(
        hard_near_train,
        0.73 * gbsa_core_oof[:, 2] + 0.16 * base_oof["cox"][48] + 0.03 * base_oof["rsf"][48] + 0.08 * near_lgb_oof[48],
        0.35 * gbsa_core_oof[:, 2] + 0.22 * base_oof["cox"][48] + 0.06 * base_oof["rsf"][48] + 0.37 * far_lgb_oof[48]
    )
    core_test_final[:, 2] = np.where(
        hard_near_test,
        0.73 * gbsa_core_test[:, 2] + 0.16 * base_test["cox"][48] + 0.03 * base_test["rsf"][48] + 0.08 * near_lgb_test[48],
        0.35 * gbsa_core_test[:, 2] + 0.22 * base_test["cox"][48] + 0.06 * base_test["rsf"][48] + 0.37 * far_lgb_test[48]
    )

    core_oof_final[:, 3] = 1.0
    core_test_final[:, 3] = 1.0
    core_oof_final = enforce_monotonicity(np.clip(core_oof_final, 0.0, 1.0))
    core_test_final = enforce_monotonicity(np.clip(core_test_final, 0.0, 1.0))
else:
    core_oof_final = exp_oof_final.copy()
    core_test_final = exp_test_final.copy()

weights_log["meta_core_mix"] = {"core_weight": 0.70, "exploratory_weight": 0.30}
oof_final = enforce_monotonicity(np.clip(0.70 * core_oof_final + 0.30 * exp_oof_final, 0.0, 1.0))
test_final = enforce_monotonicity(np.clip(0.70 * core_test_final + 0.30 * exp_test_final, 0.0, 1.0))
oof_final[:, 3] = 1.0
test_final[:, 3] = 1.0

if DO_OOF:
    exp_hybrid, exp_c_idx, exp_wb, exp_b24, exp_b48, exp_b72 = compute_hybrid_score(
        time_values, event_values, exp_oof_final[:, 1], exp_oof_final[:, 2], exp_oof_final[:, 3]
    )
    core_hybrid, core_c_idx, core_wb, core_b24, core_b48, core_b72 = compute_hybrid_score(
        time_values, event_values, core_oof_final[:, 1], core_oof_final[:, 2], core_oof_final[:, 3]
    )
    print(f"OOF Exploratory Hybrid = {exp_hybrid:.6f} | C={exp_c_idx:.6f} | WB={exp_wb:.6f}")
    print(f"OOF Core        Hybrid = {core_hybrid:.6f} | C={core_c_idx:.6f} | WB={core_wb:.6f}")

if DO_OOF:
    hybrid, c_idx, wb, b24, b48, b72 = compute_hybrid_score(
        time_values, event_values, oof_final[:, 1], oof_final[:, 2], oof_final[:, 3]
    )
    b12 = compute_brier(time_values, event_values, oof_final[:, 0], 12)
    print(f"OOF Hybrid = {hybrid:.6f}")
    print(f"OOF C-Index = {c_idx:.6f}")
    print(f"OOF WBrier = {wb:.6f}")
    print(f"OOF Brier@12 = {b12:.6f}")
    print(f"OOF Brier@24 = {b24:.6f}")
    print(f"OOF Brier@48 = {b48:.6f}")
    print(f"OOF Brier@72 = {b72:.6f}")

submission = pd.DataFrame({
    "event_id": test_df["event_id"].values,
    "prob_12h": test_final[:, 0],
    "prob_24h": test_final[:, 1],
    "prob_48h": test_final[:, 2],
    "prob_72h": test_final[:, 3],
})
submission = sample_sub[["event_id"]].merge(submission, on="event_id", how="left", validate="one_to_one")

REQUIRED_COLS = ["event_id", "prob_12h", "prob_24h", "prob_48h", "prob_72h"]

def validate_submission(sub, sample):
    required = REQUIRED_COLS
    assert list(sub.columns) == required
    assert len(sub) == len(sample)
    assert sub["event_id"].equals(sample["event_id"])
    assert sub["event_id"].nunique() == len(sub)
    vals = sub[required[1:]].values
    assert np.isfinite(vals).all()
    assert ((vals >= 0) & (vals <= 1)).all()
    assert np.all(vals[:, 0] <= vals[:, 1] + 1e-12)
    assert np.all(vals[:, 1] <= vals[:, 2] + 1e-12)
    assert np.all(vals[:, 2] <= vals[:, 3] + 1e-12)

validate_submission(submission, sample_sub)

os.makedirs(WORK_DIR, exist_ok=True)
submission.to_csv(OUTPUT_PATH, index=False)

if DO_OOF:
    oof_dump = pd.DataFrame({
        "event_id": train_df["event_id"].values,
        "prob_12h": oof_final[:, 0],
        "prob_24h": oof_final[:, 1],
        "prob_48h": oof_final[:, 2],
        "prob_72h": oof_final[:, 3],
        "event": event_values,
        "time_to_hit_hours": time_values,
    })
    oof_dump.to_csv(os.path.join(WORK_DIR, "oof_predictions.csv"), index=False)

with open(os.path.join(WORK_DIR, "blend_weights.json"), "w") as f:
    json.dump(weights_log, f, indent=2)

print("Saved:", OUTPUT_PATH)
print(submission.head())

# --- External portfolio blend over discovered CSV families ---
core_submission = submission.copy()
core_path = os.path.join(WORK_DIR, "submission_core.csv")
core_submission.to_csv(core_path, index=False)

def search_roots():
    roots = []
    for p in [DATA_DIR, WORK_DIR, ".", "/mnt/data", "/kaggle/input", "/kaggle/working"]:
        if p and os.path.isdir(p) and p not in roots:
            roots.append(p)
    return roots

SEARCH_ROOTS = search_roots()

def _extract_submission_index(name):
    name = os.path.basename(str(name))
    m = re.search(r"(?:samplecsv[_-]?)?sub(?:mission)?[_-]?(\d+)", name, flags=re.I)
    return int(m.group(1)) if m else None

def build_submission_score_lookup(roots):
    score_lookup = {}
    patterns = [
        re.compile(r"final[_-]?sub(?:mission)?[_-]?(\d+)[_-](0\.\d{4,5})", re.I),
        re.compile(r"sub(?:mission)?[_-]?(\d+)[_-](0\.\d{4,5})", re.I),
    ]
    for root in roots:
        if not os.path.isdir(root):
            continue
        for cur, _, files in os.walk(root):
            for fn in files:
                low = fn.lower()
                if not (low.endswith(".ipynb") or low.endswith(".csv") or low.endswith(".json") or low.endswith(".txt")):
                    continue
                for pat in patterns:
                    m = pat.search(fn)
                    if m:
                        sid = int(m.group(1))
                        sc = float(m.group(2))
                        if sid not in score_lookup or sc > score_lookup[sid]:
                            score_lookup[sid] = sc
    return score_lookup

SCORE_LOOKUP = build_submission_score_lookup(SEARCH_ROOTS)

def _extract_score_from_name(path, score_lookup):
    name = os.path.basename(path)
    m = re.findall(r"(?<!\d)(0\.\d{4,5})(?!\d)", name)
    if m:
        try:
            return max(float(x) for x in m)
        except Exception:
            pass
    sid = _extract_submission_index(name)
    if sid is not None and sid in score_lookup:
        return float(score_lookup[sid])
    m2 = re.findall(r"(?<!\d)(0\.\d{4,5})(?!\d)", path)
    if m2:
        try:
            return max(float(x) for x in m2)
        except Exception:
            pass
    return None

def _rank_to_core_distribution(core_vals, rank_signal):
    order = np.argsort(rank_signal, kind="mergesort")
    sorted_core = np.sort(core_vals)
    out = np.empty_like(sorted_core)
    out[order] = sorted_core
    return out

def _weighted_quantile(values, weights, q):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    if len(values) == 0:
        return 0.5
    if len(values) == 1:
        return float(values[0])
    order = np.argsort(values)
    values = values[order]
    weights = weights[order]
    total = float(weights.sum())
    if total <= 0:
        return float(np.median(values))
    cdf = np.cumsum(weights) / total
    idx = min(int(np.searchsorted(cdf, q, side="left")), len(values) - 1)
    return float(values[idx])

def _safe_read_submission_csv(path, sample_ids):
    try:
        df = pd.read_csv(path)
    except Exception:
        return None
    if list(df.columns) != REQUIRED_COLS:
        return None
    if len(df) != len(sample_ids):
        return None
    if df["event_id"].duplicated().any():
        return None
    if not df["event_id"].equals(sample_ids):
        try:
            df = sample_ids.to_frame(name="event_id").merge(df, on="event_id", how="left", validate="one_to_one")
        except Exception:
            return None
        if df.isnull().any().any():
            return None
    vals = df[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    if not np.isfinite(vals).all():
        return None
    vals = np.clip(vals, 0.0, 1.0)
    vals = enforce_monotonicity(vals)
    vals[:, 3] = np.maximum(vals[:, 3], vals[:, 2])
    df.loc[:, REQUIRED_COLS[1:]] = vals
    return df

def _hash_frame(df):
    arr = np.round(df[REQUIRED_COLS[1:]].to_numpy(dtype=float), 8)
    return hashlib.md5(arr.tobytes()).hexdigest()

def find_candidate_submission_paths():
    deny_names = {
        "train.csv", "test.csv", "sample_submission.csv",
        "submission.csv", "submission_core.csv", "submission_external.csv",
        "submission_breakthrough.csv", "submission_stage4.csv", "oof_predictions.csv"
    }
    paths = []
    seen = set()
    for root in SEARCH_ROOTS:
        if not os.path.isdir(root):
            continue
        for cur, _, files in os.walk(root):
            for fn in files:
                low = fn.lower()
                if not low.endswith(".csv"):
                    continue
                if fn in deny_names:
                    continue
                full = os.path.join(cur, fn)
                if full in seen:
                    continue
                seen.add(full)
                paths.append(full)
    return sorted(paths)

sample_ids = sample_sub["event_id"]
core_vals = core_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float)
score_ref = max(SCORE_LOOKUP.values()) if SCORE_LOOKUP else None
candidate_paths = find_candidate_submission_paths()
candidate_frames = []
candidate_meta = []

for path in candidate_paths:
    cand = _safe_read_submission_csv(path, sample_ids)
    if cand is None:
        continue
    arr = cand[REQUIRED_COLS[1:]].to_numpy(dtype=float)
    if np.allclose(arr, core_vals, atol=1e-12):
        continue
    mad_12_48 = float(np.mean(np.abs(arr[:, :3] - core_vals[:, :3])))
    if (not np.isfinite(mad_12_48)) or mad_12_48 < 1e-7 or mad_12_48 > 0.08:
        continue
    corrs = []
    for j in range(3):
        x = arr[:, j]
        y = core_vals[:, j]
        if np.std(x) < 1e-12 or np.std(y) < 1e-12:
            corrs.append(1.0)
        else:
            corrs.append(float(np.corrcoef(x, y)[0, 1]))
    candidate_frames.append(cand)
    candidate_meta.append({
        "path": path,
        "sub_id": _extract_submission_index(path),
        "score": _extract_score_from_name(path, SCORE_LOOKUP),
        "mad_12_48": mad_12_48,
        "mean_corr_12_48": float(np.mean(corrs)),
        "hash": _hash_frame(cand),
    })

seen_hash = set()
dedup_frames, dedup_meta = [], []
for df, meta in zip(candidate_frames, candidate_meta):
    if meta["hash"] in seen_hash:
        continue
    seen_hash.add(meta["hash"])
    dedup_frames.append(df)
    dedup_meta.append(meta)
candidate_frames, candidate_meta = dedup_frames, dedup_meta

if candidate_frames and score_ref is not None:
    score_gap_keep = 0.0125
    kept_frames, kept_meta = [], []
    for df, meta in zip(candidate_frames, candidate_meta):
        s = meta["score"]
        if (s is None) or (s >= score_ref - score_gap_keep):
            kept_frames.append(df)
            kept_meta.append(meta)
    candidate_frames, candidate_meta = kept_frames, kept_meta

if candidate_frames:
    order = sorted(
        range(len(candidate_meta)),
        key=lambda i: (
            candidate_meta[i]["score"] is not None,
            candidate_meta[i]["score"] if candidate_meta[i]["score"] is not None else -1e9,
            candidate_meta[i]["mad_12_48"],
        ),
        reverse=True,
    )[:18]
    candidate_frames = [candidate_frames[i] for i in order]
    candidate_meta = [candidate_meta[i] for i in order]

if candidate_frames:
    candidate_arrays = [df[REQUIRED_COLS[1:]].to_numpy(dtype=float) for df in candidate_frames]
    family_eps = 3e-4
    picked = []
    for i in range(len(candidate_meta)):
        arr_i = candidate_arrays[i][:, :3]
        too_close = False
        for j in picked:
            mad_pair = float(np.mean(np.abs(arr_i - candidate_arrays[j][:, :3])))
            if mad_pair < family_eps:
                too_close = True
                break
        if not too_close:
            picked.append(i)
    candidate_frames = [candidate_frames[i] for i in picked]
    candidate_meta = [candidate_meta[i] for i in picked]

if candidate_frames:
    candidate_arrays = np.stack([df[REQUIRED_COLS[1:]].to_numpy(dtype=float) for df in candidate_frames], axis=0)
    mads = np.array([m["mad_12_48"] for m in candidate_meta], dtype=float)
    mad_med = max(float(np.median(mads)), 1e-6)

    raw_weights = []
    for meta in candidate_meta:
        s = meta["score"]
        if (score_ref is None) or (s is None):
            score_w = 0.85
        else:
            score_w = float(np.exp((s - score_ref) / 0.0025))
        div_w = float(np.clip(np.sqrt(meta["mad_12_48"] / mad_med), 0.75, 1.35))
        raw_weights.append(score_w * div_w)
    raw_weights = np.asarray(raw_weights, dtype=float)
    weights = raw_weights / raw_weights.sum()

    prob_mean = np.tensordot(weights, candidate_arrays, axes=(0, 0))
    prob_median = np.zeros_like(prob_mean)
    prob_q25 = np.zeros_like(prob_mean)
    prob_q75 = np.zeros_like(prob_mean)
    for i in range(prob_mean.shape[0]):
        for j in range(prob_mean.shape[1]):
            prob_median[i, j] = _weighted_quantile(candidate_arrays[:, i, j], weights, 0.50)
            prob_q25[i, j] = _weighted_quantile(candidate_arrays[:, i, j], weights, 0.25)
            prob_q75[i, j] = _weighted_quantile(candidate_arrays[:, i, j], weights, 0.75)

    consensus = 0.55 * prob_mean + 0.45 * prob_median

    rankblend = np.zeros_like(prob_mean)
    for j in range(4):
        rank_signal = np.tensordot(
            weights,
            np.stack([
                df[REQUIRED_COLS[1:]].rank(method="average", pct=True).to_numpy(dtype=float)[:, j]
                for df in candidate_frames
            ], axis=0),
            axes=(0, 0),
        )
        rankblend[:, j] = _rank_to_core_distribution(core_vals[:, j], rank_signal)

    spread_band = prob_q75 - prob_q25
    band_scale = np.array([0.020, 0.025, 0.030, 0.020], dtype=float)
    agreement = np.exp(-spread_band / band_scale)

    n_cand = len(candidate_meta)
    strength_scale = min(1.0, 0.25 + 0.18 * n_cand)
    known_frac = float(np.mean([m["score"] is not None for m in candidate_meta]))
    known_scale = 0.85 + 0.15 * known_frac

    prob_alpha = np.array([0.09, 0.15, 0.23, 0.00], dtype=float) * strength_scale * known_scale
    rank_alpha = np.array([0.03, 0.06, 0.10, 0.00], dtype=float) * strength_scale * known_scale
    max_total = np.array([0.14, 0.24, 0.34, 0.00], dtype=float)

    zone_factor = np.column_stack([
        np.where(dist_test < 3500, 0.60, np.where(dist_test < 9000, 1.00, 0.82)),
        np.where(dist_test < 4000, 0.68, np.where(dist_test < 12000, 1.00, 0.90)),
        np.where(dist_test < 4500, 0.76, np.where(dist_test < 16000, 1.00, 0.96)),
        np.zeros(len(dist_test), dtype=float),
    ])

    uncertainty = 1.0 - np.abs(2.0 * core_vals - 1.0)
    phys_ref = np.column_stack([
        0.60 * anchor_test[12] + 0.40 * timing_test[12],
        0.35 * anchor_test[24] + 0.65 * timing_test[24],
        0.25 * anchor_test[48] + 0.75 * timing_test[48],
        np.ones(len(dist_test), dtype=float),
    ])
    direction_bonus = 1.0 + 0.15 * (((consensus - core_vals) * (phys_ref - core_vals)) > 0).astype(float)

    final_vals = core_vals.copy()
    for j in range(4):
        a = prob_alpha[j] * agreement[:, j] * zone_factor[:, j] * direction_bonus[:, j]
        b = rank_alpha[j] * agreement[:, j] * zone_factor[:, j] * (0.85 + 0.30 * uncertainty[:, j])
        if j < 3:
            a *= (0.75 + 0.50 * uncertainty[:, j])
        total_ab = a + b
        shrink = np.minimum(1.0, max_total[j] / np.maximum(total_ab, 1e-9))
        a *= shrink
        b *= shrink
        keep = np.clip(1.0 - a - b, 0.55, 1.0)
        final_vals[:, j] = keep * core_vals[:, j] + a * consensus[:, j] + b * rankblend[:, j]

    final_vals = np.clip(final_vals, 0.0, 1.0)
    final_vals = enforce_monotonicity(final_vals)
    final_vals[:, 3] = 1.0

    external_submission = core_submission.copy()
    external_submission.loc[:, REQUIRED_COLS[1:]] = final_vals
    validate_submission(external_submission, sample_sub)

    external_path = os.path.join(WORK_DIR, "submission_external.csv")
    breakthrough_path = os.path.join(WORK_DIR, "submission_breakthrough.csv")
    external_submission.to_csv(external_path, index=False)
    external_submission.to_csv(breakthrough_path, index=False)
    external_submission.to_csv(OUTPUT_PATH, index=False)

    with open(os.path.join(WORK_DIR, "external_blend_summary.json"), "w") as f:
        json.dump({
            "score_lookup": SCORE_LOOKUP,
            "score_ref": score_ref,
            "n_candidates": len(candidate_meta),
            "weights": [
                {
                    "path": m["path"],
                    "sub_id": m["sub_id"],
                    "score": m["score"],
                    "mad_12_48": m["mad_12_48"],
                    "mean_corr_12_48": m["mean_corr_12_48"],
                    "weight": float(w),
                }
                for m, w in zip(candidate_meta, weights)
            ],
            "blend": {
                "prob_alpha": prob_alpha.tolist(),
                "rank_alpha": rank_alpha.tolist(),
                "max_total": max_total.tolist(),
                "strength_scale": float(strength_scale),
                "known_scale": float(known_scale),
            }
        }, f, indent=2)

    print("External candidates used:", len(candidate_meta))
    for m, w in sorted(zip(candidate_meta, weights), key=lambda z: -z[1])[:20]:
        print(f"  w={w:.4f} | score={m['score']} | mad={m['mad_12_48']:.6f} | corr={m['mean_corr_12_48']:.6f} | {m['path']}")
    print("Saved:", external_path)
    print("Saved:", breakthrough_path)
    print("Saved:", OUTPUT_PATH)
else:
    core_submission.to_csv(os.path.join(WORK_DIR, "submission_external.csv"), index=False)
    core_submission.to_csv(os.path.join(WORK_DIR, "submission_breakthrough.csv"), index=False)
    core_submission.to_csv(OUTPUT_PATH, index=False)
    with open(os.path.join(WORK_DIR, "external_blend_summary.json"), "w") as f:
        json.dump({
            "score_lookup": SCORE_LOOKUP,
            "score_ref": score_ref,
            "n_candidates": 0,
            "weights": [],
            "blend": None,
        }, f, indent=2)
    print("No external CSV candidates found. submission.csv = submission_core.csv")
    print("Saved:", OUTPUT_PATH)

# --- Stage-4: conservative discrete-hazard + analog refinement ---
if "weights_log" not in globals():
    weights_log = {}

try:
    stage4_base_submission = pd.read_csv(OUTPUT_PATH)
    validate_submission(stage4_base_submission, sample_sub)
except Exception:
    if "core_submission" in globals():
        stage4_base_submission = core_submission.copy()
    else:
        stage4_base_submission = submission.copy()

base_vals_stage4 = stage4_base_submission[REQUIRED_COLS[1:]].to_numpy(dtype=float)
base_oof_stage4 = oof_final.copy() if "oof_final" in globals() else np.tile(base_vals_stage4.mean(axis=0), (len(train_df), 1))
stage4_seeds = choose_seeds(BASE_SEEDS[:6]) if ("choose_seeds" in globals() and "BASE_SEEDS" in globals()) else (123, 456, 789, 2024)

intervals_stage4 = [(0.0, 12.0), (12.0, 24.0), (24.0, 48.0)]

STAGE4_DTH_FEATURES = [c for c in [
    "dist_km", "log_distance", "inv_distance", "fire_radius_km", "area_to_dist_ratio",
    "effective_closing_speed", "eta_effective", "log_eta_effective", "alignment_abs",
    "threat_score_eff", "num_perimeters_0_5h", "dt_first_last_0_5h",
    "low_temporal_resolution_0_5h", "near_speed_rank", "far_threat_rank",
    "dist_fit_r2_0_5h", "slope_toward", "accel_toward", "growth_intensity",
    "speed_alignment", "effective_alignment", "zone_near", "zone_warning", "zone_far",
    "hour_sin", "hour_cos", "month_sin", "month_cos"
] if c in train_proc.columns]

STAGE4_KNN_FEATURES = [c for c in [
    "dist_km", "eta_effective", "alignment_abs", "fire_radius_km", "num_perimeters_0_5h",
    "dt_first_last_0_5h", "low_temporal_resolution_0_5h", "threat_score_eff",
    "growth_intensity", "near_speed_rank", "far_threat_rank", "dist_fit_r2_0_5h",
    "hour_sin", "hour_cos", "month_sin", "month_cos"
] if c in train_proc.columns]

def _stage4_make_pp_observed(df_proc, owner_idx, feature_cols):
    rows, owners, y, interval_ids = [], [], [], []
    for i in owner_idx:
        t = float(time_values[i])
        e = int(event_values[i])
        base_row = df_proc.iloc[i]
        for k, (lo, hi) in enumerate(intervals_stage4):
            if t <= lo + 1e-12:
                break
            if (e == 0) and (t < hi - 1e-12):
                break
            row = base_row[feature_cols].to_dict()
            row["interval_id"] = k
            row["interval_mid"] = 0.5 * (lo + hi)
            row["interval_hi"] = hi
            row["interval_width"] = hi - lo
            row["log_interval_hi"] = np.log1p(hi)
            row["is_late_interval"] = 1.0 if hi > 24 else 0.0
            row["risk_x_interval"] = float(base_row.get("threat_score_eff", 0.0)) * (k + 1.0)
            row["eta_gap"] = float(base_row.get("eta_effective", 9999.0)) - hi
            rows.append(row)
            owners.append(i)
            interval_ids.append(k)
            hit = int((e == 1) and (t > lo) and (t <= hi))
            y.append(hit)
            if hit == 1:
                break
    return pd.DataFrame(rows), np.asarray(y, dtype=int), np.asarray(owners, dtype=int), np.asarray(interval_ids, dtype=int)

def _stage4_make_pp_predict(df_proc, owner_idx, feature_cols):
    rows = []
    for i in owner_idx:
        base_row = df_proc.iloc[i]
        for k, (lo, hi) in enumerate(intervals_stage4):
            row = base_row[feature_cols].to_dict()
            row["interval_id"] = k
            row["interval_mid"] = 0.5 * (lo + hi)
            row["interval_hi"] = hi
            row["interval_width"] = hi - lo
            row["log_interval_hi"] = np.log1p(hi)
            row["is_late_interval"] = 1.0 if hi > 24 else 0.0
            row["risk_x_interval"] = float(base_row.get("threat_score_eff", 0.0)) * (k + 1.0)
            row["eta_gap"] = float(base_row.get("eta_effective", 9999.0)) - hi
            rows.append(row)
    return pd.DataFrame(rows)

def _stage4_hazards_to_cum(pred_haz, n_obs):
    out = np.zeros((n_obs, 4), dtype=float)
    ptr = 0
    for i in range(n_obs):
        h0 = float(np.clip(pred_haz[ptr + 0], 1e-6, 1.0 - 1e-6))
        h1 = float(np.clip(pred_haz[ptr + 1], 1e-6, 1.0 - 1e-6))
        h2 = float(np.clip(pred_haz[ptr + 2], 1e-6, 1.0 - 1e-6))
        s0 = 1.0 - h0
        s1 = s0 * (1.0 - h1)
        s2 = s1 * (1.0 - h2)
        out[i, 0] = 1.0 - s0
        out[i, 1] = 1.0 - s1
        out[i, 2] = 1.0 - s2
        out[i, 3] = 1.0
        ptr += 3
    return enforce_monotonicity(np.clip(out, 0.0, 1.0))

def _stage4_pred1(model, X):
    p = model.predict_proba(X)
    p = np.asarray(p)
    return p[:, 1] if p.ndim == 2 else p

def _stage4_build_lgb(seed):
    try:
        return lgb.LGBMClassifier(
            n_estimators=360,
            learning_rate=0.03,
            num_leaves=15,
            min_child_samples=18,
            subsample=0.82,
            colsample_bytree=0.78,
            reg_alpha=0.40,
            reg_lambda=2.25,
            random_state=seed,
            objective="binary",
            verbosity=-1,
        )
    except Exception:
        return None

def _stage4_build_cat(seed):
    if "CatBoostClassifier" not in globals():
        return None
    try:
        return CatBoostClassifier(
            iterations=420,
            depth=5,
            learning_rate=0.03,
            loss_function="Logloss",
            eval_metric="Logloss",
            random_seed=seed,
            verbose=False,
            l2_leaf_reg=5.0,
            subsample=0.85,
            bootstrap_type="Bernoulli",
            allow_writing_files=False,
        )
    except Exception:
        return None

def _stage4_build_lr(seed):
    try:
        return Pipeline([
            ("scaler", StandardScaler()),
            ("lr", LogisticRegression(C=0.65, max_iter=800))
        ])
    except Exception:
        return None

def _stage4_fit_dth(train_proc, test_proc, feature_cols, seeds):
    n_train, n_test = len(train_proc), len(test_proc)
    oof = np.zeros((n_train, 4), dtype=float)
    cnt = np.zeros(n_train, dtype=float)
    test_acc = np.zeros((n_test, 4), dtype=float)
    full_test = np.zeros((n_test, 4), dtype=float)

    test_pp = _stage4_make_pp_predict(test_proc, np.arange(n_test), feature_cols)

    for seed in seeds:
        cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
        seed_test = np.zeros((n_test, 4), dtype=float)

        for tr_idx, va_idx in cv.split(train_proc, event_values):
            pp_tr, y_tr, _, int_tr = _stage4_make_pp_observed(train_proc, tr_idx, feature_cols)
            if len(pp_tr) == 0 or len(np.unique(y_tr)) < 2:
                continue

            row_w = np.where(int_tr == 0, 0.90, np.where(int_tr == 1, 1.05, 1.30))
            row_w = row_w * np.where(y_tr == 1, 1.25, 1.0)

            pp_va = _stage4_make_pp_predict(train_proc, va_idx, feature_cols)

            pred_va_parts = []
            pred_test_parts = []

            for build in (_stage4_build_lgb, _stage4_build_cat, _stage4_build_lr):
                model = build(seed)
                if model is None:
                    continue
                try:
                    if isinstance(model, Pipeline):
                        last_name = model.steps[-1][0]
                        model.fit(pp_tr, y_tr, **{f"{last_name}__sample_weight": row_w})
                    else:
                        model.fit(pp_tr, y_tr, sample_weight=row_w)
                except Exception:
                    try:
                        model.fit(pp_tr, y_tr)
                    except Exception:
                        continue
                try:
                    pred_va_parts.append(_stage4_pred1(model, pp_va))
                    pred_test_parts.append(_stage4_pred1(model, test_pp))
                except Exception:
                    pass

            if pred_va_parts:
                haz_va = np.mean(pred_va_parts, axis=0)
                haz_test = np.mean(pred_test_parts, axis=0)
                fold_pred = _stage4_hazards_to_cum(haz_va, len(va_idx))
                oof[va_idx] += fold_pred
                cnt[va_idx] += 1.0
                seed_test += _stage4_hazards_to_cum(haz_test, n_test) / 5.0

        test_acc += seed_test / max(len(seeds), 1)

    full_fill = np.zeros((n_train, 4), dtype=float)
    full_pp_train = _stage4_make_pp_predict(train_proc, np.arange(n_train), feature_cols)

    for seed in seeds:
        pp_tr_all, y_tr_all, _, int_tr_all = _stage4_make_pp_observed(train_proc, np.arange(n_train), feature_cols)
        if len(pp_tr_all) == 0 or len(np.unique(y_tr_all)) < 2:
            continue

        row_w_all = np.where(int_tr_all == 0, 0.90, np.where(int_tr_all == 1, 1.05, 1.30))
        row_w_all = row_w_all * np.where(y_tr_all == 1, 1.25, 1.0)

        pred_train_parts = []
        pred_test_parts = []

        for build in (_stage4_build_lgb, _stage4_build_cat, _stage4_build_lr):
            model = build(seed)
            if model is None:
                continue
            try:
                if isinstance(model, Pipeline):
                    last_name = model.steps[-1][0]
                    model.fit(pp_tr_all, y_tr_all, **{f"{last_name}__sample_weight": row_w_all})
                else:
                    model.fit(pp_tr_all, y_tr_all, sample_weight=row_w_all)
            except Exception:
                try:
                    model.fit(pp_tr_all, y_tr_all)
                except Exception:
                    continue
            try:
                pred_train_parts.append(_stage4_pred1(model, full_pp_train))
                pred_test_parts.append(_stage4_pred1(model, test_pp))
            except Exception:
                pass

        if pred_train_parts:
            full_fill += _stage4_hazards_to_cum(np.mean(pred_train_parts, axis=0), n_train) / max(len(seeds), 1)
            full_test += _stage4_hazards_to_cum(np.mean(pred_test_parts, axis=0), n_test) / max(len(seeds), 1)

    nz = cnt > 0
    if nz.any():
        oof[nz] /= cnt[nz, None]
    if (~nz).any():
        oof[~nz] = full_fill[~nz]

    zero_test = np.isclose(test_acc.sum(axis=1), 0.0)
    if zero_test.any():
        test_acc[zero_test] = full_test[zero_test]

    return enforce_monotonicity(np.clip(oof, 0.0, 1.0)), enforce_monotonicity(np.clip(test_acc, 0.0, 1.0))

def _stage4_knn_prob(train_mat, train_y, query_mat, k=11, power=2.0):
    if len(train_mat) == 0:
        return np.full(len(query_mat), 0.5, dtype=float)
    if len(train_mat) == 1:
        return np.full(len(query_mat), float(train_y[0]), dtype=float)
    k = int(max(1, min(k, len(train_mat))))
    diff = query_mat[:, None, :] - train_mat[None, :, :]
    dist = np.sqrt(np.maximum((diff * diff).sum(axis=2), 1e-12))
    idx = np.argpartition(dist, kth=k - 1, axis=1)[:, :k]
    dsel = np.take_along_axis(dist, idx, axis=1)
    ysel = train_y[idx]
    w = 1.0 / np.power(dsel + 1e-6, power)
    return np.sum(w * ysel, axis=1) / np.sum(w, axis=1)

def _stage4_fit_knn(train_proc, test_proc, feature_cols, seeds):
    Xtr = train_proc[feature_cols].to_numpy(dtype=float)
    Xte = test_proc[feature_cols].to_numpy(dtype=float)

    med = np.median(Xtr, axis=0)
    iqr = np.quantile(Xtr, 0.75, axis=0) - np.quantile(Xtr, 0.25, axis=0)
    iqr = np.where(iqr < 1e-6, 1.0, iqr)

    Xtr = (Xtr - med) / iqr
    Xte = (Xte - med) / iqr

    if Xtr.shape[1] > 0:
        boost_cols = []
        for name in ["dist_km", "eta_effective", "alignment_abs", "threat_score_eff"]:
            if name in feature_cols:
                boost_cols.append(feature_cols.index(name))
        if boost_cols:
            Xtr[:, boost_cols] *= 1.35
            Xte[:, boost_cols] *= 1.35

    oof = np.zeros((len(train_proc), 4), dtype=float)
    test = np.zeros((len(test_proc), 4), dtype=float)

    for col_idx, h in enumerate([12, 24, 48]):
        y_h, mask_h = make_binary_target(time_values, event_values, h)
        vidx = np.where(mask_h)[0]
        if len(vidx) < 8:
            continue

        oof_h = np.zeros(len(train_proc), dtype=float)
        cnt_h = np.zeros(len(train_proc), dtype=float)

        for seed in seeds:
            cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
            Xv = Xtr[vidx]
            yv = y_h[vidx]
            for tr_sub, va_sub in cv.split(Xv, yv):
                tr_idx = vidx[tr_sub]
                va_idx = vidx[va_sub]
                preds = []
                for k in (7, 11, 15):
                    preds.append(_stage4_knn_prob(Xtr[tr_idx], y_h[tr_idx], Xtr[va_idx], k=k))
                oof_h[va_idx] += np.mean(preds, axis=0)
                cnt_h[va_idx] += 1.0

        full_preds = []
        for k in (7, 11, 15, 21):
            full_preds.append(_stage4_knn_prob(Xtr[vidx], y_h[vidx], Xtr, k=k))
        full_fill = np.mean(full_preds, axis=0)

        nz = cnt_h > 0
        if nz.any():
            oof_h[nz] /= cnt_h[nz]
        if (~nz).any():
            oof_h[~nz] = full_fill[~nz]

        test_preds = []
        for k in (7, 11, 15, 21):
            test_preds.append(_stage4_knn_prob(Xtr[vidx], y_h[vidx], Xte, k=k))
        test_h = np.mean(test_preds, axis=0)

        oof[:, col_idx] = np.clip(oof_h, 0.0, 1.0)
        test[:, col_idx] = np.clip(test_h, 0.0, 1.0)

    oof[:, 3] = 1.0
    test[:, 3] = 1.0
    return enforce_monotonicity(oof), enforce_monotonicity(test)

stage4_components_oof = []
stage4_components_test = []
stage4_component_weights = []

try:
    dth_oof, dth_test = _stage4_fit_dth(train_proc, test_proc, STAGE4_DTH_FEATURES, stage4_seeds[:4])
    stage4_components_oof.append(dth_oof)
    stage4_components_test.append(dth_test)
    stage4_component_weights.append(0.55)
    print("Stage-4 DTH ready.")
except Exception as e:
    print("[WARN] Stage-4 DTH unavailable:", e)

try:
    knn_oof, knn_test = _stage4_fit_knn(train_proc, test_proc, STAGE4_KNN_FEATURES, stage4_seeds[:4])
    stage4_components_oof.append(knn_oof)
    stage4_components_test.append(knn_test)
    stage4_component_weights.append(0.20)
    print("Stage-4 KNN ready.")
except Exception as e:
    print("[WARN] Stage-4 KNN unavailable:", e)

try:
    phys_train_signal = (
        -np.log1p(train_proc["eta_effective"].values)
        + 0.18 * train_proc["alignment_abs"].values
        + 0.10 * train_proc["near_speed_rank"].values
        + 0.08 * train_proc["threat_rank_global"].values
        + 0.06 * train_proc["dist_fit_r2_0_5h"].values
    )
    phys_test_signal = (
        -np.log1p(test_proc["eta_effective"].values)
        + 0.18 * test_proc["alignment_abs"].values
        + 0.10 * test_proc["near_speed_rank"].values
        + 0.08 * test_proc["threat_rank_global"].values
        + 0.06 * test_proc["dist_fit_r2_0_5h"].values
    )
    phys_oof = np.zeros((len(train_df), 4), dtype=float)
    phys_test = np.zeros((len(test_df), 4), dtype=float)
    for col_idx, h in enumerate([12, 24, 48]):
        y_h, mask_h = make_binary_target(time_values, event_values, h)
        poof, ptest = repeated_isotonic(phys_train_signal, phys_test_signal, y_h, mask_h, stage4_seeds[:4], n_splits=5)
        phys_oof[:, col_idx] = poof
        phys_test[:, col_idx] = ptest
    phys_oof[:, 3] = 1.0
    phys_test[:, 3] = 1.0
    phys_oof = enforce_monotonicity(np.clip(phys_oof, 0.0, 1.0))
    phys_test = enforce_monotonicity(np.clip(phys_test, 0.0, 1.0))
    stage4_components_oof.append(phys_oof)
    stage4_components_test.append(phys_test)
    stage4_component_weights.append(0.25)
    print("Stage-4 physics prior ready.")
except Exception as e:
    print("[WARN] Stage-4 physics prior unavailable:", e)

if stage4_components_oof:
    w_stage4 = np.asarray(stage4_component_weights, dtype=float)
    w_stage4 = w_stage4 / w_stage4.sum()
    aux_oof_stage4 = np.tensordot(w_stage4, np.stack(stage4_components_oof, axis=0), axes=(0, 0))
    aux_test_stage4 = np.tensordot(w_stage4, np.stack(stage4_components_test, axis=0), axes=(0, 0))
    aux_oof_stage4 = enforce_monotonicity(np.clip(aux_oof_stage4, 0.0, 1.0))
    aux_test_stage4 = enforce_monotonicity(np.clip(aux_test_stage4, 0.0, 1.0))
else:
    aux_oof_stage4 = base_oof_stage4.copy()
    aux_test_stage4 = base_vals_stage4.copy()

base_stage4_score, _, _, _, _, _ = compute_hybrid_score(
    time_values, event_values, base_oof_stage4[:, 1], base_oof_stage4[:, 2], base_oof_stage4[:, 3]
)
aux_stage4_score, _, _, _, _, _ = compute_hybrid_score(
    time_values, event_values, aux_oof_stage4[:, 1], aux_oof_stage4[:, 2], aux_oof_stage4[:, 3]
)

near_train = dist_train < 5000
near_test = dist_test < 5000
grid_near = [0.00, 0.04, 0.08, 0.12, 0.16, 0.20, 0.24]
grid_far = [0.00, 0.04, 0.08, 0.12, 0.16]

best_cfg = None
best_score_stage4 = -1e18

for a24n in grid_near:
    for a24f in grid_far:
        for a48n in grid_near:
            for a48f in grid_near:
                cand = base_oof_stage4.copy()
                cand[near_train, 1] = (1.0 - a24n) * cand[near_train, 1] + a24n * aux_oof_stage4[near_train, 1]
                cand[~near_train, 1] = (1.0 - a24f) * cand[~near_train, 1] + a24f * aux_oof_stage4[~near_train, 1]
                cand[near_train, 2] = (1.0 - a48n) * cand[near_train, 2] + a48n * aux_oof_stage4[near_train, 2]
                cand[~near_train, 2] = (1.0 - a48f) * cand[~near_train, 2] + a48f * aux_oof_stage4[~near_train, 2]
                cand[:, 0] = 0.82 * cand[:, 0] + 0.18 * aux_oof_stage4[:, 0]
                cand = enforce_monotonicity(np.clip(cand, 0.0, 1.0))
                cand[:, 3] = 1.0
                sc, _, _, _, _, _ = compute_hybrid_score(
                    time_values, event_values, cand[:, 1], cand[:, 2], cand[:, 3]
                )
                sc -= 0.0018 * (a24n + a24f + a48n + a48f)
                if sc > best_score_stage4:
                    best_score_stage4 = sc
                    best_cfg = (a24n, a24f, a48n, a48f)

if best_cfg is None:
    best_cfg = (0.0, 0.0, 0.0, 0.0)

a24n, a24f, a48n, a48f = best_cfg
stage4_final = base_vals_stage4.copy()
stage4_final[near_test, 1] = (1.0 - a24n) * stage4_final[near_test, 1] + a24n * aux_test_stage4[near_test, 1]
stage4_final[~near_test, 1] = (1.0 - a24f) * stage4_final[~near_test, 1] + a24f * aux_test_stage4[~near_test, 1]
stage4_final[near_test, 2] = (1.0 - a48n) * stage4_final[near_test, 2] + a48n * aux_test_stage4[near_test, 2]
stage4_final[~near_test, 2] = (1.0 - a48f) * stage4_final[~near_test, 2] + a48f * aux_test_stage4[~near_test, 2]
stage4_final[:, 0] = 0.82 * stage4_final[:, 0] + 0.18 * aux_test_stage4[:, 0]
stage4_final = enforce_monotonicity(np.clip(stage4_final, 0.0, 1.0))
stage4_final[:, 3] = 1.0

stage4_submission = stage4_base_submission.copy()
stage4_submission.loc[:, REQUIRED_COLS[1:]] = stage4_final
validate_submission(stage4_submission, sample_sub)
stage4_path = os.path.join(WORK_DIR, "submission_stage4.csv")
stage4_submission.to_csv(stage4_path, index=False)
stage4_submission.to_csv(OUTPUT_PATH, index=False)

weights_log["stage4"] = {
    "n_components": int(len(stage4_components_oof)),
    "component_weights": [float(x) for x in (w_stage4.tolist() if stage4_components_oof else [])],
    "base_hybrid": float(base_stage4_score),
    "aux_hybrid": float(aux_stage4_score),
    "best_hybrid_reg": float(best_score_stage4),
    "cfg": {
        "a24_near": float(a24n),
        "a24_far": float(a24f),
        "a48_near": float(a48n),
        "a48_far": float(a48f),
    },
}
with open(os.path.join(WORK_DIR, "blend_weights.json"), "w") as f:
    json.dump(weights_log, f, indent=2)

print("Stage-4 base hybrid:", base_stage4_score)
print("Stage-4 aux hybrid :", aux_stage4_score)
print("Stage-4 best cfg   :", best_cfg, "score_reg =", best_score_stage4)
print("Saved:", stage4_path)
print("Saved:", OUTPUT_PATH)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.0/4.0 MB 38.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.9/8.9 MB 85.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 222.1/222.1 kB 14.7 MB/s eta 0:00:00
DATA_DIR = /kaggle/input/competitions/WiDSWorldWide_GlobalDathon26
TRAIN = (221, 37) TEST = (95, 35) HAS_SKSURV = True HAS_CAT = True
OOF Exploratory Hybrid = 0.970239 | C=0.935989 | WB=0.015083
OOF Core        Hybrid = 0.971879 | C=0.938639 | WB=0.013876
OOF Hybrid = 0.972454
OOF C-Index = 0.940378
OOF WBrier = 0.013799
OOF Brier@12 = 0.052256
OOF Brier@24 = 0.026482
OOF Brier@48 = 0.014635
OOF Brier@72 = 0.000000
Saved: /kaggle/working/submission.csv
   event_id  prob_12h  prob_24h  prob_48h  prob_72h
0  10662602  0.017776  0.022023  0.022023       1.0
1  13353600  0.673829  0.932840  0.962051       1.0
2  13942327  0.017125  0.021231  0.021231       1.0
3  16112781  0.777136  0.949012  0.972149       1.0
4  17132808  0.027851  0.032738  0.038736       